
# 01 | Victorian Council Data Ingestion

## Project
Victorian Local Government Financial Sustainability
& Performance Intelligence (2020–2025)

## Purpose
Victorian councils publish performance data every year.
This notebook combines five years of council data
into one dataset for further analysis.

## Tools
- Databricks
- PySpark
- Delta Lake

## What this notebook does
1. Reads five annual CSV files.
2. Combines them into one dataset.
3. Checks the number of records.
4. Saves the data in a Bronze Delta table.
5. Exports an optional CSV copy.

## Expected output
- 23,068 records
- 5 financial years
- 10 columns
- Bronze table: workspace.default.bronze_performance

## Next step
Check data quality and clean the records
before calculating KPIs in Power BI.


In [0]:

from pyspark.sql import functions as F
from functools import reduce

# Folder where the original files are stored
folder = "/Volumes/workspace/default/council_raw"

# Files we want to combine
files = {
    "2020-21": "Indicators 2020-21.csv",
    "2021-22": "Indicators 2021-22.csv",
    "2022-23": "Indicators 2022-23.csv",
    "2023-24": "Indicators 2023-24.csv",
    "2024-25": "Indicators 2024-25.csv"
}

# Store each year's data here
dataframes = []

for year, filename in files.items():

    # Read the CSV file
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"{folder}/{filename}")
    )

    # Add the financial year
    df = df.withColumn("FinancialYear", F.lit(year))

    # Add this year's data to our list
    dataframes.append(df)

    print(year, ":", df.count(), "rows")

# Combine all five years
all_indicators = reduce(
    lambda a, b: a.unionByName(
        b, allowMissingColumns=True
    ),
    dataframes
)

print("Total rows:", all_indicators.count())
print("Total columns:", len(all_indicators.columns))

display(all_indicators.limit(10))


2020-21 : 4582 rows
2021-22 : 4582 rows
2022-23 : 4582 rows
2023-24 : 4661 rows
2024-25 : 4661 rows
Total rows: 23068
Total columns: 10


Council,Group,ID,Description,Service Provided,Data applicable?,Result,Variation,FinancialYear,Target
Southern Grampians Shire,LARGE_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1.205349439,Officer response times have improved due to responsiveness to animal management requests.,2020-21,null
South Gippsland Shire,LARGE_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,2.04040404,"Animal management requests were impacted due to the Victorian Government COVID-19 Pandemic restrictions, with limited payment options due to office closure, and not being able to attend properties in person.",2020-21,null
Stonnington City,METRO,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1.80608365,There has been a slight increase in response time this financial year due to several factors: - Increased number of animal registrations in Stonnington leading to a higher volume of requests. - Continued Covid-19 workplace closures have impacted on efficiency. An overall pleasing result that is within our expected service standard.,2020-21,null
Ararat Rural City,SMALL_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1,"Council places a strong emphasis on responsible pet ownership and community safety, ensuring all requests for animal management received are actioned on the same day.",2020-21,null
Strathbogie Shire,SMALL_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1.69982238,"Council has employed dedicated local laws staff and there were some issues with the animal pound. All of these are now resolved, and further improvement is expected.",2020-21,null
Surf Coast Shire,LARGE_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1.350993377,Council has reviewed the method for calculating this indicator and this has resulted in the high variance from last year. 2019-20 reporting was inaccurate.,2020-21,null
Swan Hill Rural City,LARGE_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1.074164629,The decrease in response time in 2021 was achieved by having a full complement of staff.,2020-21,null
Towong Shire,SMALL_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1.945945946,"There were 37 animal management requests in 2020/21 compared to 46 in 2019/20, 49 in 2018/19 and 53 in 2017/18. Towong Shire Council had two part time Rangers in 2020/21 compared to one part-time ranger and an on-call contractor in prior years.The nature of a request and when it is received determine how quickly it is actioned.",2020-21,null
Wellington Shire,LARGE_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1.17572892,"Local Laws officers responded to 1,269 domestic animal management requests during the reporting period, with an average initial response time of 1.18 days. The response time was slightly higher than the last reporting period due in part to COVID restrictions requiring staff to work remotely.",2020-21,null
West Wimmera Shire,SMALL_SHIRE,AM1,Average number of days it has taken for council to action animal management requests,YES,applicable,1,Information for 2019/20 was not available due to multiple changes in staffing across the year unfortunately resulting in consistent records not being kept. Council has maintained records during 2020/21 and took 1 day to action each animal management request during the year.,2020-21,null


In [0]:

# Expected records for each year
expected = {
    "2020-21": 4582,
    "2021-22": 4582,
    "2022-23": 4582,
    "2023-24": 4661,
    "2024-25": 4661
}

# Count the records in our combined data
year_counts = all_indicators.groupBy(
    "FinancialYear"
).count()

display(year_counts.orderBy("FinancialYear"))

# Convert counts into a Python dictionary
actual = {
    row["FinancialYear"]: row["count"]
    for row in year_counts.collect()
}

# Stop if the data is different from expected
assert actual == expected, "Yearly row counts do not match"
assert all_indicators.count() == 23068, "Total rows do not match"
assert len(all_indicators.columns) == 10, "Column count does not match"

print("PASS: All five years loaded correctly")


FinancialYear,count
2020-21,4582
2021-22,4582
2022-23,4582
2023-24,4661
2024-25,4661


PASS: All five years loaded correctly


In [0]:

# Rename two columns for easier use in Delta Lake
bronze = (
    all_indicators
    .withColumnRenamed(
        "Service Provided", "Service_Provided"
    )
    .withColumnRenamed(
        "Data applicable?", "Data_Applicable"
    )
)

# Save the combined data
bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.bronze_performance")

# Read the saved table to verify it
saved = spark.table("workspace.default.bronze_performance")

print("Saved rows:", saved.count())
print("Saved columns:", len(saved.columns))

assert saved.count() == 23068
assert len(saved.columns) == 10

print("PASS: Bronze table saved correctly")

display(saved.limit(10))


Saved rows: 23068
Saved columns: 10
PASS: Bronze table saved correctly


Council,Group,ID,Description,Service_Provided,Data_Applicable,Result,Variation,FinancialYear,Target
Alpine Shire,Small Shire,AF2,Health inspections of aquatic facilities,YES,Applicable,0,"Council did not conduct any health inspections of aquatic facilities during 2022/23, due to a lack of Environmental Health staff. Council made the Department of Health aware of its continued efforts to recruit Environmental Health staff. Water testing was undertaken at Bright and Porepunkah River pools during the peak summer holiday period.",2022-23,null
Alpine Shire,Small Shire,AF6,Utilisation of aquatic facilities,YES,Applicable,2.352789374,"Overall visitation to aquatic facilities declined in 2022/23 compared to the previous year. The decline was primarily driven by almost 6,400 fewer visits across the Mount Beauty and Myrtleford swimming pools caused by wet weather experienced during October – December 2022, however this was partially offset by an increase in visitation to the Bright Sports Centre in the second half of 2022/23.",2022-23,null
Alpine Shire,Small Shire,AF7,Cost of aquatic facilities,YES,Applicable,19.03684109,Apparent cost per visit has increased due to the reduction in visitation during 2022/23 compared to the previous year. Actual costs have reduced slightly compared to the previous year.,2022-23,null
Alpine Shire,Small Shire,AM1,Time taken to action animal management requests,YES,Applicable,2.535353535,"The total number of animal management jobs increased by 23% compared to the previous year. Of these, 73% received a first response action within 1 day, 9% within 2 days, and 9% within 5 days of receiving the request. A small proportion (4%) of jobs took longer than 10 days for a first response action, which are responsible for the apparent increase in time taken to respond in 2022/23. Animal management cases can take 4-5 times longer to bring to completion than dealing with other Local Laws tasks such as a parking complaint.",2022-23,null
Alpine Shire,Small Shire,AM2,Animals reclaimed,YES,Applicable,0.4875,"Council collected slightly fewer domestic animals in 2022/23 compared to the previous year. The community continue to use social media to assist in returning lost animals to their owners, meaning Council officers are generally called where these social media efforts are unsuccessful - which can impact on the number of animals returned to their owners.",2022-23,null
Alpine Shire,Small Shire,AM5,Animals rehomed,YES,Applicable,0.3375,"While the number of animals rehomed was slightly fewer than the previous year, when animals returned to their owners and animals rehomed are considered together, 83% of domestic animals collected by Council found their way to a home in 2022/23. This is an improvement compared to the previous year. Council's agreement with the Wangaratta Animal Shelter will come to an end on 31 July 2023, when the responsibility for rehoming of animals will revert to Council.",2022-23,null
Alpine Shire,Small Shire,AM6,Cost of animal management service per population,YES,Applicable,9.983301708,"Cost per registered animal has increased slightly compared to the previous year, but is still less than the average cost for Small Rural Councils in 2021/22.",2022-23,null
Alpine Shire,Small Shire,AM7,Animal management prosecutions,YES,Applicable,0,No animal management prosecutions were presented to Court in 2022/23.,2022-23,null
Alpine Shire,Small Shire,FS1,Time taken to action food complaints,YES,Applicable,3.727272727,"Council was without permanent Environmental Health staff from June 2022 onwards, which impacted on the time taken to respond to food complaints. Despite this, 64% of food complaints received a first response action within two days.",2022-23,null
Alpine Shire,Small Shire,FS2,Food safety assessments,YES,Applicable,0.783216783,"Council undertook the majority of assessments in the first half of 2022, prior to the departure of Environmental Health staff in June 2022. Council is continuing its efforts to recr

In [0]:

# Read the Bronze table
bronze = spark.table("workspace.default.bronze_performance")

# Where the CSV copy will be saved
export_path = (
    "/Volumes/workspace/default/council_exports/"
    "combined_2020_2025"
)

# Export with correct handling of quotation marks
bronze.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("quoteAll", True) \
    .csv(export_path)

print("CSV export completed")


CSV export completed



## Results and Outcomes

### What I completed
- Combined five years of Victorian council performance data.
- Validated 23,068 records across five financial years.
- Created a reusable Bronze Delta table.
- Exported an optional CSV copy.

### Final Output
Table: workspace.default.bronze_performance

Records: 23,068
Columns: 10
Period: 2020–21 to 2024–25

### Validation Results
- Annual record counts: PASS
- Bronze Delta table: PASS
- CSV export: Completed (not independently verified)

### Business Value
Created a consolidated source for analysing
council performance trends and benchmarking
across Victorian councils.

### Next Step
Notebook 02 — Council Data Quality

Check missing values, duplicates and numeric
data before preparing the Silver dataset.
